# 04 · Construct research variables

**Module:** 2 · **Time:** 45 minutes · **Prerequisites:** Module 2: cleaning

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ningzhanglab/Python-for-Beginners/blob/main/modules/02_data_wrangling/04_creating_variables.ipynb) · [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ningzhanglab/Python-for-Beginners/main?filepath=modules%2F02_data_wrangling%2F04_creating_variables.ipynb)

**In Colab choose File → Save a copy in Drive before editing.** The repository is private and badges point to `main` after merge.

## Learning goals

- **C**reate readable numeric variables from messy wage strings.
- **C**onstruct indicators, bins, and logarithms for empirical work.
- **U**se vectorization and method chaining instead of row-by-row loops.

In [ ]:
import numpy as np
import pandas as pd

survey = pd.DataFrame({
    "person_id": [1001, 1002, 1003, 1003, 1005, 1006],
    "household_id": ["H001", "H001", "H002", "H002", "H003", "H003"],
    "region": [" North ", "SOUTH", "east", "north", "West", "SOUTH"],
    "employment": ["employed", "unemployed", "employed", "employed", "employed", "unemployed"],
    "wage_2023": ["40000", "-99", "51000", "52000", "72000", "not reported"],
    "wage_2024": ["42000", "28000", "53000", "54000", "9999999", "30000"],
    "survey_weight": [1.2, 0.8, 1.1, 1.4, 0.9, 1.6],
})
print("Fictional observations:", len(survey))

## Why this matters

Regression-ready data need clean, well-documented measures. Creating a variable is not just renaming a column: its units, missingness, and substantive definition matter. Here we work only with fictional wage records.

## Create valid numeric wages

Parse strings explicitly, treat `-99` as missing, and flag extreme toy values rather than silently dropping records.

In [ ]:
prepared = survey.copy()
wage23 = pd.to_numeric(prepared["wage_2023"], errors="coerce")
wage24 = pd.to_numeric(prepared["wage_2024"], errors="coerce")
prepared["wage23"] = wage23.mask(wage23.eq(-99))
prepared["wage24"] = wage24.mask(wage24.gt(500_000))
print(prepared[["wage23", "wage24"]])

## Indicators and categories

`np.where` creates a two-way indicator; `pd.cut` bins a numeric variable. Boundaries and treatment of missing observations must be stated clearly.

In [ ]:
prepared["employed"] = prepared["employment"].eq("employed").astype(int)
prepared["income_band"] = pd.cut(
    prepared["wage24"], bins=[0, 40_000, 70_000, float("inf")],
    labels=["lower", "middle", "higher"]
)
print(prepared[["employed", "income_band"]])

## Log wages and growth

Only log positive numbers. When a value is missing or invalid, keep the derived measure missing too. In larger datasets, vectorized operations are usually more efficient than loops.

In [ ]:
prepared["ln_wage24"] = np.log(prepared["wage24"])
prepared["growth_pct"] = (prepared["wage24"] / prepared["wage23"] - 1) * 100
print(prepared[["ln_wage24", "growth_pct"]].head())

## Use assign and method chaining

`assign` returns a new DataFrame. The method chain keeps multiple transformations together, making their order visible.

In [ ]:
mini = (survey.assign(
    region_clean=lambda frame: frame["region"].str.strip().str.lower(),
    employed=lambda frame: frame["employment"].eq("employed")
).loc[:, ["person_id", "region_clean", "employed"]])
print(mini.head())

## If you know Stata/R

| Task | pandas | Stata | R |
| --- | --- | --- | --- |
| New variable | `df['z'] = ...` | `gen z = ...` | `mutate(df, z = ...)` |
| Indicator | `s.eq('yes').astype(int)` | `gen yes = x == "yes"` | `as.integer(x == 'yes')` |
| Bins | `pd.cut(s, bins=...)` | `recode` / `egen cut()` | `cut(s, breaks=...)` |
| Log | `np.log(s)` | `ln(s)` | `log(s)` |

For binary indicators, explicitly decide what a missing source value should mean; missing is not automatically zero.

## Try it

Complete the answer cells, then run the friendly checks. Blank learner exercises produce hints rather than exceptions.

### ★ Exercise 1 — Numeric 2024 wage

Create `wage24_numeric` by converting `survey['wage_2024']` to numbers (invalid text becomes missing).

In [ ]:
wage24_numeric = None

In [ ]:
expected = pd.to_numeric(survey["wage_2024"], errors="coerce")
if isinstance(wage24_numeric, pd.Series) and wage24_numeric.equals(expected):
    print("✓ Numeric wages ready")
else:
    print("Hint: pd.to_numeric(..., errors='coerce') parses the strings.")

### ★★ Exercise 2 — Employment indicator

Assign `employed_flag` as an integer Series: 1 if employed, 0 otherwise.

In [ ]:
employed_flag = None

In [ ]:
expected = survey["employment"].eq("employed").astype(int)
if isinstance(employed_flag, pd.Series) and employed_flag.equals(expected):
    print("✓ Indicator correct")
else:
    print("Hint: compare employment with 'employed', then convert to int.")

### ★★ Exercise 3 — Cleaned log wage

Create `log_wage24` as log numeric wages, with the fictional outlier above 500,000 marked missing.

In [ ]:
log_wage24 = None

In [ ]:
expected = np.log(pd.to_numeric(survey["wage_2024"], errors="coerce").mask(
    lambda values: values.gt(500_000)
))
if isinstance(log_wage24, pd.Series) and np.allclose(log_wage24, expected, equal_nan=True):
    print("✓ Log wage correct")
else:
    print("Hint: exclude the toy outlier before applying np.log.")

### ★★★ Exercise 4 — Build a clean region column with assign

Create `assigned_regions` as a DataFrame based on `survey`, adding `region_clean` with stripped lowercase text. Keep every row.

In [ ]:
assigned_regions = None

In [ ]:
expected = survey.assign(region_clean=lambda frame: frame["region"].str.strip().str.lower())
if isinstance(assigned_regions, pd.DataFrame) and assigned_regions.equals(expected):
    print("✓ Assigned region column correct")
else:
    print("Hint: use assign with a lambda that reads the region column.")

## Common errors

- A log of zero or a negative wage is undefined; check the domain before using `np.log`.
- Forgetting the `-99` missing code creates impossible percentage growth.
- `pd.cut` bins have boundary conventions; inspect values at thresholds.
- A row-wise `apply` is often unnecessary when vectorized operations exist.

## Recap

- Document numeric parsing, missing rules and units.
- Use `np.where`/Boolean comparisons and `pd.cut` to construct variables.
- Prefer transparent vectorized transformations and method chains.
- Check what happens to missing and extreme source values.

## Next steps

Next: [Merging household and person data](05_merging.ipynb). Reference: [pandas assigning columns](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html).

[Report a content error](https://github.com/ningzhanglab/Python-for-Beginners/issues/new?template=content_error.md).